# 🏥 Hospital Readmission Prediction (UCI Diabetes Dataset)

This project builds an interpretable machine learning model to predict hospital readmissions using the UCI Diabetes 130-US hospitals dataset. It applies real-world techniques for class imbalance, model evaluation, and interpretability using SHAP.

## 1. Introduction

Hospital readmissions are a major concern in healthcare due to their cost and potential to reflect preventable issues in care delivery. In this project, we aim to build a machine learning model that can predict whether a diabetic patient will be readmitted to the hospital using the UCI Diabetes dataset.

We will:
- Clean and prepare over 100K patient records
- Handle significant class imbalance in the target variable
- Train an XGBoost model for classification
- Evaluate performance using recall and precision-recall curves
- Use SHAP for model interpretability

This notebook walks through the end-to-end workflow from data preparation to model explanation.

In [1]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling & evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, roc_auc_score
from xgboost import XGBClassifier, plot_importance
import shap

# Load the data
df = pd.read_csv("diabetic_data.csv")

# Preview shape and first few rows
print(f"Dataset shape: {df.shape}")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'diabetic_data.csv'

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.drop(['weight', 'encounter_id', 'patient_nbr', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3'], axis=1, inplace=True)


## 2. Data Loading & Initial Exploration

We load the UCI Diabetes dataset containing 100K+ records across 130 hospitals. First, we inspect its shape, basic statistics, and class distribution for the `readmitted` column.

`## 3. Data Cleaning

We clean the dataset to ensure it is ready for machine learning:

- Drop or encode ID fields
- Replace placeholder missing values (`'?'`) with `NaN`
- Encode or drop low-utility columns
- Check and handle missing values
- Prepare categorical columns for encoding


In [ ]:
# Check for placeholder '?' values
(df == '?').sum().sort_values(ascending=False)

In [ ]:
# Actual missing values (NaNs)
df.isna().sum().sort_values(ascending=False)

In [ ]:
# Raw counts
df['readmitted'].value_counts()

In [ ]:
df.gender.value_counts()


In [ ]:
# 1. One-hot encode 'gender' and assign to X
df = pd.get_dummies(df, columns=['gender'])

# 2. Ensure all expected dummy columns exist (safe even if a value is missing)
for col in ['gender_Female', 'gender_Male', 'gender_Unknown/Invalid']:
    if col not in df.columns:
        df[col] = 0

# 3. Cast the dummy columns to int
df[['gender_Female', 'gender_Male', 'gender_Unknown/Invalid']] = df[['gender_Female', 'gender_Male', 'gender_Unknown/Invalid']].astype(int)
df.info()

In [ ]:
df.race.value_counts()

In [ ]:
df['race'] = df['race'].replace('?', np.nan)

In [ ]:
df = pd.get_dummies(df, columns=['race'], prefix='race')

## 3. Data Cleaning

In [ ]:
df.info()

In [ ]:
df.age.value_counts()

In [ ]:
age_map = {
    '[0-10)'   : 0,
    '[10-20)'  : 1,
    '[20-30)'  : 2,
    '[30-40)'  : 3,
    '[40-50)'  : 4,
    '[50-60)'  : 5,
    '[60-70)'  : 6,
    '[70-80)'  : 7,
    '[80-90)'  : 8,
    '[90-100)' : 9
}

df['age'] = df['age'].map(age_map)

In [ ]:
df.age.value_counts()

In [ ]:
df.info()

In [ ]:
drug_columns = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide', 'examide',
    'citoglipton', 'insulin', 'glyburide-metformin',
    'glipizide-metformin', 'glimepiride-pioglitazone',
    'metformin-rosiglitazone', 'metformin-pioglitazone'
]

for col in drug_columns:
    df[col] = df[col].apply(lambda x: 0 if x == 'No' else 1)

## 4. Feature Engineering

In [ ]:
df['change'] = df['change'].apply(lambda x: 1 if x == 'Ch' else 0)
df['diabetesMed'] = df['diabetesMed'].apply(lambda x: 1 if x == 'Yes' else 0)

In [ ]:
glu_map = {'None': 0, 'Norm': 1, '>200': 2, '>300': 3}
df['max_glu_serum'] = df['max_glu_serum'].map(glu_map)

a1c_map = {'None': 0, 'Norm': 1, '>7': 2, '>8': 3}
df['A1Cresult'] = df['A1Cresult'].map(a1c_map)

In [ ]:
df['max_glu_serum'] = df['max_glu_serum'].fillna(0)
df['A1Cresult'] = df['A1Cresult'].fillna(0)

In [ ]:
# Convert 'readmitted' to binary: 1 if <30 days, 0 otherwise
df['readmitted_binary'] = df['readmitted'].apply(lambda x: 1 if x == '<30' else 0)
df.drop(columns=['readmitted'], inplace=True)
# Show updated class distribution
df['readmitted_binary'].value_counts()


In [ ]:
df.info()

In [ ]:
# If you have the full DataFrame (e.g., df), do this:
y = df['readmitted_binary']
X = df.drop(columns=['readmitted_binary'])

In [ ]:
print(y.unique())  # Should show: [0 1 2]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Optional: Drop NaNs just in case
mask = y_train.notna()
X_train = X_train[mask]
y_train = y_train[mask]

## 5. Modeling with XGBoost

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(eval_metric='logloss')
model.fit(X_train, y_train)

In [ ]:
print(model)  

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)
accuracy_score(y_test, y_pred)

In [ ]:
print(y_test.value_counts(normalize=True))

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred, target_names=['Not Readmitted', 'Readmitted']))

In [ ]:
unique, counts = np.unique(y_pred, return_counts=True)
print(dict(zip(unique, counts)))

## 6. Evaluation

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

# Get feature importances
importances = rf.feature_importances_
feature_importance = pd.Series(importances, index=X_train.columns).sort_values(ascending=False)
print(feature_importance.head(15))


In [ ]:
# ratio of negative / positive examples
ratio = (y_train == 0).sum() / (y_train == 1).sum()

model = XGBClassifier(
    scale_pos_weight=ratio,
    eval_metric='logloss')
model.fit(X_train, y_train)


In [ ]:
y_pred = model.predict(X_test)
accuracy_score(y_test, y_pred)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, model.predict(X_test), target_names=["Not Readmitted", "Readmitted"]))


## 7. Model Interpretability with SHAP

In [ ]:
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt

y_probs = model.predict_proba(X_test)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, y_probs)

plt.plot(recall, precision)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.grid(True)
plt.show()


In [ ]:
from xgboost import plot_importance
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plot_importance(model, max_num_features=15, importance_type='gain')  # or 'weight' or 'cover'
plt.title("Top 15 Feature Importances (by Gain)")
plt.show()


In [ ]:
import shap

explainer = shap.Explainer(model)
shap_values = explainer(X_test)

shap.plots.beeswarm(shap_values, max_display=15)


## 8. Conclusion

Final remarks and potential future work directions.